# V3 Feature Extraction: Lexical & Multilingual Transformer Embeddings


## 1. Imports & Environment Configuration


In [1]:
import os
import re
import gc
import time
import numpy as np
import pandas as pd
from unidecode import unidecode
from rapidfuzz import fuzz
import torch
from sentence_transformers import SentenceTransformer

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"PyTorch Version: {torch.__version__}")
print(f"Compute Device:  {device}")
print(f"RapidFuzz Version: {fuzz.__file__}")


PyTorch Version: 2.12.0
Compute Device:  mps
RapidFuzz Version: /opt/homebrew/lib/python3.11/site-packages/rapidfuzz/fuzz.py



## 2. Load Entity Data & Candidate Pairs


In [2]:
# Load sample of candidate pairs and entity tables for feature matrix extraction
s1_path = 'dataset/test/test_source1.tsv'
s2_path = 'dataset/test/test_source2.tsv'
s3_path = 'dataset/test/test_source3.tsv'
cand_path = 'output/candidate_pairs.tsv'

# Read entity lookups
print("Loading entity tables...")
df_s1 = pd.read_csv(s1_path, sep='\t', nrows=10000)
df_s2 = pd.read_csv(s2_path, sep='\t', nrows=50000)
df_s3 = pd.read_csv(s3_path, sep='\t', nrows=50000)

s1_lookup = {row['entity_id']: (str(row['business_name']), str(row['business_address']), str(row['country'])) for _, row in df_s1.iterrows()}
target_lookup = {}
for _, row in df_s2.iterrows():
    target_lookup[row['entity_id']] = (str(row['business_name']), str(row['business_address']), str(row['country']), 1.0)
for _, row in df_s3.iterrows():
    target_lookup[row['entity_id']] = (str(row['business_name']), str(row['business_address']), str(row['country']), 0.0)

# Sample candidate pairs for matrix generation
candidate_samples = []
with open(cand_path, 'r', encoding='utf-8') as f:
    next(f) # header
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2 and parts[1]:
            s1_id = parts[0]
            cands = parts[1].split(',')
            if s1_id in s1_lookup:
                for c_id in cands[:5]: # top-5 candidates per entity sample
                    if c_id in target_lookup:
                        candidate_samples.append((s1_id, c_id))
        if len(candidate_samples) >= 1000:
            break

df_pairs = pd.DataFrame(candidate_samples, columns=['source1_id', 'candidate_id'])
print(f"Sample Candidate Pairs to Process: {len(df_pairs):,}")
print(df_pairs.head())


Loading entity tables...
Sample Candidate Pairs to Process: 1,000
     source1_id  candidate_id
0  S1-280204013  S2-965581679
1  S1-778640743  S3-467616579
2  S1-378191895   S2-21480418
3  S1-502376041  S2-551281247
4  S1-339015166  S2-376340088



## 3. Lexical Feature Extraction (RapidFuzz)


In [3]:
def extract_lexical_features(s1_name, s1_addr, s1_ctry, c_name, c_addr, c_ctry):
    # Transliterated lower strings
    n1 = unidecode(str(s1_name)).lower()
    n2 = unidecode(str(c_name)).lower()
    a1 = unidecode(str(s1_addr)).lower()
    a2 = unidecode(str(c_addr)).lower()
    
    # 1. Name Lexical Ratios
    fuzz_ratio = fuzz.ratio(n1, n2) / 100.0
    token_set = fuzz.token_set_ratio(n1, n2) / 100.0
    token_sort = fuzz.token_sort_ratio(n1, n2) / 100.0
    partial_ratio = fuzz.partial_ratio(n1, n2) / 100.0
    len_diff = abs(len(n1) - len(n2))
    
    # 2. Address Lexical Ratios
    addr_ratio = fuzz.ratio(a1, a2) / 100.0
    addr_token_set = fuzz.token_set_ratio(a1, a2) / 100.0
    
    # 3. Numeric & PIN Matches
    pins1 = set(re.findall(r'\b\d{5,6}\b', a1))
    pins2 = set(re.findall(r'\b\d{5,6}\b', a2))
    pin_match = 1.0 if (pins1 and pins2 and pins1.intersection(pins2)) else 0.0
    
    nums1 = set(re.findall(r'\b\d+\b', a1))
    nums2 = set(re.findall(r'\b\d+\b', a2))
    if nums1 and nums2:
        inter = nums1.intersection(nums2)
        num_match = 1.0 if inter else 0.0
    else:
        num_match = 0.5
        
    country_match = 1.0 if str(s1_ctry).strip() == str(c_ctry).strip() else 0.0
    
    return [
        fuzz_ratio,
        token_set,
        token_sort,
        partial_ratio,
        len_diff,
        addr_ratio,
        addr_token_set,
        pin_match,
        num_match,
        country_match
    ]

lex_features = []
for _, row in df_pairs.iterrows():
    s1_info = s1_lookup[row['source1_id']]
    c_info = target_lookup[row['candidate_id']]
    feats = extract_lexical_features(s1_info[0], s1_info[1], s1_info[2], c_info[0], c_info[1], c_info[2])
    feats.append(c_info[3]) # is_s2 target source indicator
    lex_features.append(feats)

lex_cols = [
    'name_fuzz_ratio', 'name_token_set_ratio', 'name_token_sort_ratio', 'name_partial_ratio',
    'name_len_diff', 'addr_fuzz_ratio', 'addr_token_set_ratio', 'pin_match', 'num_match',
    'country_match', 'is_s2'
]
df_lex = pd.DataFrame(lex_features, columns=lex_cols)
print(f"Extracted {len(df_lex)} Lexical Feature Rows:")
print(df_lex.head())


Extracted 1000 Lexical Feature Rows:
   name_fuzz_ratio  name_token_set_ratio  ...  country_match  is_s2
0         0.292683              0.243902  ...            1.0    1.0
1         0.878049              1.000000  ...            1.0    0.0
2         0.640000              0.640000  ...            1.0    1.0
3         0.830189              1.000000  ...            1.0    1.0
4         0.454545              0.500000  ...            1.0    1.0

[5 rows x 11 columns]



## 4. Multilingual Transformer Embeddings (multilingual-e5-large-instruct)


In [4]:
# Load multilingual-e5-large-instruct
model_name = 'intfloat/multilingual-e5-large-instruct'
print(f"Loading {model_name}...")
t0 = time.time()
transformer_model = SentenceTransformer(model_name, device=device)
print(f"Model loaded in {time.time() - t0:.2f}s")

# Extract unique text strings for batched embedding inference
s1_unique_ids = df_pairs['source1_id'].unique()
c_unique_ids = df_pairs['candidate_id'].unique()

# Format queries with instruction prefix as specified by e5-large-instruct
s1_texts_name = [f"Instruct: Given an entity name, retrieve matching representations.\nQuery: {s1_lookup[uid][0]}" for uid in s1_unique_ids]
s1_texts_full = [f"Instruct: Given an entity name and address, retrieve matching representations.\nQuery: {s1_lookup[uid][0]} {s1_lookup[uid][1]}" for uid in s1_unique_ids]

c_texts_name = [target_lookup[uid][0] for uid in c_unique_ids]
c_texts_full = [f"{target_lookup[uid][0]} {target_lookup[uid][1]}" for uid in c_unique_ids]

print(f"Encoding {len(s1_unique_ids):,} Query Entities...")
s1_name_emb = transformer_model.encode(s1_texts_name, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
s1_full_emb = transformer_model.encode(s1_texts_full, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Encoding {len(c_unique_ids):,} Target Entities...")
c_name_emb = transformer_model.encode(c_texts_name, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
c_full_emb = transformer_model.encode(c_texts_full, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

s1_name_map = dict(zip(s1_unique_ids, s1_name_emb))
s1_full_map = dict(zip(s1_unique_ids, s1_full_emb))
c_name_map = dict(zip(c_unique_ids, c_name_emb))
c_full_map = dict(zip(c_unique_ids, c_full_emb))

print("Embeddings computed and mapped successfully.")


Loading intfloat/multilingual-e5-large-instruct...
Model loaded in 6.21s
Encoding 879 Query Entities...
Encoding 986 Target Entities...
Embeddings computed and mapped successfully.

Batches: 100%|##########| 16/16 [00:02<00:00,  5.35it/s]



## 5. Compute Cosine Similarities & Assemble Feature Matrix


In [5]:
# Compute Cosine Similarities (Dot product of L2-normalized embeddings)
cos_sim_name = []
cos_sim_full = []

for _, row in df_pairs.iterrows():
    u_n = s1_name_map[row['source1_id']]
    v_n = c_name_map[row['candidate_id']]
    cos_sim_name.append(float(np.dot(u_n, v_n)))
    
    u_f = s1_full_map[row['source1_id']]
    v_f = c_full_map[row['candidate_id']]
    cos_sim_full.append(float(np.dot(u_f, v_f)))

# Assemble Unified Feature Matrix
df_features = df_lex.copy()
df_features['e5_name_cosine_sim'] = cos_sim_name
df_features['e5_full_cosine_sim'] = cos_sim_full
df_features.insert(0, 'candidate_id', df_pairs['candidate_id'])
df_features.insert(0, 'source1_id', df_pairs['source1_id'])

print("Unified Feature Matrix Assembly Completed:")
print(f"Total Rows:     {len(df_features):,}")
print(f"Total Features: {df_features.shape[1] - 2}")
print("\nFeature Correlation with e5_name_cosine_sim:")
print(df_features.select_dtypes(include=[np.number]).corr()['e5_name_cosine_sim'].sort_values(ascending=False))
print("\nFeature Matrix Head:")
print(df_features.head(10))


Unified Feature Matrix Assembly Completed:
Total Rows:     1,000
Total Features: 13

Feature Correlation with e5_name_cosine_sim:
e5_name_cosine_sim       1.000000
name_token_set_ratio     0.835147
name_fuzz_ratio          0.817615
name_partial_ratio       0.817141
name_token_sort_ratio    0.803456
e5_full_cosine_sim       0.618985
addr_fuzz_ratio          0.031843
addr_token_set_ratio     0.029182
is_s2                   -0.048216
pin_match               -0.113620
num_match               -0.278786
name_len_diff           -0.326449
country_match                 NaN
Name: e5_name_cosine_sim, dtype: float64

Feature Matrix Head:
     source1_id  candidate_id  ...  e5_name_cosine_sim  e5_full_cosine_sim
0  S1-280204013  S2-965581679  ...            0.759766            0.890625
1  S1-778640743  S3-467616579  ...            0.906250            0.944824
2  S1-378191895   S2-21480418  ...            0.844238            0.891113
3  S1-502376041  S2-551281247  ...            0.925293           

## 6. Export Feature Matrix


In [6]:
output_dir = 'output/v3_features'
os.makedirs(output_dir, exist_ok=True)

parquet_path = os.path.join(output_dir, 'v3_features_sample.parquet')
df_features.to_parquet(parquet_path, index=False)

matrix_X = df_features.drop(columns=['source1_id', 'candidate_id']).values
npy_path = os.path.join(output_dir, 'v3_feature_matrix_sample.npy')
np.save(npy_path, matrix_X)

print(f"Saved Parquet Features: {parquet_path} ({os.path.getsize(parquet_path) / 1024:.2f} KB)")
print(f"Saved NumPy Feature Matrix: {npy_path} ({os.path.getsize(npy_path) / 1024:.2f} KB)")
print(f"Feature Matrix Shape: {matrix_X.shape}")


Saved Parquet Features: output/v3_features/v3_features_sample.parquet (62.62 KB)
Saved NumPy Feature Matrix: output/v3_features/v3_feature_matrix_sample.npy (101.69 KB)
Feature Matrix Shape: (1000, 13)

